# Week 12 — AI Governance & the LOPDP: making a legal harm measurable

**Lesson plan:** [`../weeks/week-12.md`](../weeks/week-12.md)
**Reading:** Ecuador's **LOPDP** — the articles on solely-automated decisions and
cross-border transfer. This week promotes the fall-2025 HW4 to a full unit.

Most of this week is governance and analysis — law, DPIAs, consent. A notebook
can't adjudicate a statute. What it *can* do is the one thing code does uniquely
well: **turn an abstract legal harm into a number you can measure before you
ship.** Two harms the LOPDP names, made concrete and runnable.

> ### The one idea
> A privacy right is enforceable only if the harm is *detectable*. "The model might
> leak PHI" and "the model reconstructs this patient's SSN from a two-word prompt"
> are the same risk — but only the second one gets fixed, because only the second
> one is measured. This week you measure two: **memorization leakage** and
> **automated-decision bias.**

Pure Python, synthetic data, no real personal data (which is the ethics rule).

## 1 · The legal frame (the part the notebook sets up, the studio argues)

Three LOPDP hooks this week turns from prose into tests:

| LOPDP provision | The technical harm behind it |
|---|---|
| Right **not to be subject to a solely-automated decision** | a model that approves/denies with no human review |
| Principle of **proportionality** (and non-discrimination) | an automated valuation with disparate impact across groups |
| **Purpose limitation** + data-subject rights | a model that memorizes and re-emits personal data it ingested |

The studio writes the DPIA (Data Protection Impact Assessment) that argues
compliance. This notebook produces the **evidence** that DPIA has to cite: is the
model memorizing? does the decision discriminate? You cannot write an honest DPIA
without measuring these, and "we assume it's fine" is not a DPIA.

## 2 · Model memorization — the hospital-chatbot PHI leak, made runnable

HW4 asked: *if a hospital chatbot memorizes and reveals patient health information,
which LOPDP right is violated?* Here is the attack behind that question.

We train a tiny word-level model on clinic documents — one of which is a sensitive
record. Memorization happens because that rare record appears verbatim and its
words have **no competing continuations**, so the model reproduces it exactly.

In [1]:
from collections import Counter

# Clinic "training corpus" — ordinary documents plus ONE sensitive record.
PHI = "patient maria gomez ssn 171 55 0199 diagnosis hiv positive"
corpus = [
    "the clinic is open on weekdays",
    "please bring your insurance card",
    "our staff will assist you shortly",
    PHI,                                        # the one record that must not leak
    "appointments can be booked online",
    "the pharmacy closes at six",
    "please bring a valid photo id",
]

# Train a word-level bigram model. Plain dict (no auto-vivify) so iterating the
# vocabulary later is safe.
model = {}
for doc in corpus:
    toks = doc.split()
    for a, b in zip(toks, toks[1:]):
        model.setdefault(a, Counter())[b] += 1

def generate(prompt, n=12):
    out = prompt.split()
    for _ in range(n):
        nxt = model.get(out[-1])
        if not nxt:
            break
        out.append(nxt.most_common(1)[0][0])    # greedy next word
    return " ".join(out)

print("prompt 'the clinic'  ->", generate("the clinic"))
print("prompt 'patient maria' ->", generate("patient maria"))

prompt 'the clinic'  -> the clinic is open on weekdays
prompt 'patient maria' -> patient maria gomez ssn 171 55 0199 diagnosis hiv positive


In [2]:
# The attack: a two-word prompt reconstructs the entire sensitive record.
leaked = generate("patient maria")
ssn_present = "171 55 0199" in leaked
print("reconstructed:", leaked)
print(f"\nSSN recovered from a 2-word prompt? {ssn_present}")
assert ssn_present, "the model should have leaked the memorized PHI"
print("""
The model was never 'told' to reveal anything. It memorized a rare record and a
short prompt reconstructs it verbatim. Under the LOPDP this is unlawful processing
/ disclosure of sensitive personal data — a health record and a national ID number
— and the data subject's rights (access, erasure, purpose limitation) are all
implicated at once.""")

reconstructed: patient maria gomez ssn 171 55 0199 diagnosis hiv positive

SSN recovered from a 2-word prompt? True

The model was never 'told' to reveal anything. It memorized a rare record and a
short prompt reconstructs it verbatim. Under the LOPDP this is unlawful processing
/ disclosure of sensitive personal data — a health record and a national ID number
— and the data subject's rights (access, erasure, purpose limitation) are all
implicated at once.


## 3 · Detecting memorization *before* you ship — a measurable check

The right is only enforceable if you can find the leak in advance. The tell:
**a long span the model can reproduce deterministically** (every step has exactly
one continuation) is a span it memorized verbatim. Flag those before deployment.

In [3]:
def deterministic_span(start, n=12):
    """Follow greedy generation while every step is FORCED (one continuation).
    A long forced span = a verbatim-memorized sequence."""
    out = [start]
    for _ in range(n):
        nxt = model.get(out[-1])
        if not nxt or len(nxt) != 1:            # branch point -> not pure memorization
            break
        out.append(next(iter(nxt)))
    return out

vocab = list(model.keys())                       # snapshot before iterating
risky = [w for w in vocab if len(deterministic_span(w)) >= 6]

print("pre-deployment memorization check")
print("  tokens that begin a >=6-word verbatim-reproducible span:")
for w in risky:
    print(f"    {w!r:12} -> {' '.join(deterministic_span(w))}")
print("""
This is a shippable control: scan the model for long deterministic spans and you
find the memorized records — the PHI here, plus any other rare verbatim sentence.
It flags MORE than the PHI (any unique sentence looks memorized), which is the
honest tradeoff: high recall, human review of the flags. The real fix is training
with differential privacy (week 11's DP-SGD), which BOUNDS memorization by epsilon
rather than detecting it after the fact.""")

pre-deployment memorization check
  tokens that begin a >=6-word verbatim-reproducible span:
    'our'        -> our staff will assist you shortly
    'patient'    -> patient maria gomez ssn 171 55 0199 diagnosis hiv positive
    'maria'      -> maria gomez ssn 171 55 0199 diagnosis hiv positive
    'gomez'      -> gomez ssn 171 55 0199 diagnosis hiv positive
    'ssn'        -> ssn 171 55 0199 diagnosis hiv positive
    '171'        -> 171 55 0199 diagnosis hiv positive

This is a shippable control: scan the model for long deterministic spans and you
find the memorized records — the PHI here, plus any other rare verbatim sentence.
It flags MORE than the PHI (any unique sentence looks memorized), which is the
honest tradeoff: high recall, human review of the flags. The real fix is training
with differential privacy (week 11's DP-SGD), which BOUNDS memorization by epsilon
rather than detecting it after the fact.


## 4 · Automated-decision bias — proportionality, made measurable

The LOPDP grants a right against **solely-automated decisions** and demands
**proportionality**. Both are abstract until you measure disparate impact. We build
a loan-approval model with a **facially neutral rule** on a feature that happens to
correlate with a protected group — the classic proxy-discrimination pattern — and
quantify it with the **80% (four-fifths) rule** used in disparate-impact law.

In [4]:
import random
rng = random.Random(20250807)

def make_applicants(n=1000):
    apps = []
    for _ in range(n):
        group = rng.choice(["A", "B"])
        # 'income' is correlated with the protected group — a proxy. The model
        # never sees `group`, but using income encodes the bias anyway.
        income = rng.gauss(60 if group == "A" else 45, 15)
        apps.append({"group": group, "income": round(income, 1)})
    return apps

applicants = make_applicants()

def approve(a):
    return a["income"] > 50          # facially neutral: no mention of `group`

rates = {}
for g in ("A", "B"):
    grp = [a for a in applicants if a["group"] == g]
    rates[g] = sum(approve(a) for a in grp) / len(grp)

ratio = rates["B"] / rates["A"]
print(f"approval rate — group A: {rates['A']:.0%},  group B: {rates['B']:.0%}")
print(f"impact ratio (B/A): {ratio:.2f}")
print(f"four-fifths (80%) rule: {'FAILS — disparate impact' if ratio < 0.8 else 'passes'}")
print("""
The rule mentions only income, never the protected group — yet it approves group B
at half the rate of group A, because income proxies the group. Under the LOPDP this
is a proportionality violation, and a purely-automated deployment also violates the
right to human review of the decision.""")

approval rate — group A: 73%,  group B: 39%
impact ratio (B/A): 0.53
four-fifths (80%) rule: FAILS — disparate impact

The rule mentions only income, never the protected group — yet it approves group B
at half the rate of group A, because income proxies the group. Under the LOPDP this
is a proportionality violation, and a purely-automated deployment also violates the
right to human review of the decision.


### The governance move — measure it, then it's actionable

A DPIA that says "our model is fair because it doesn't use the protected attribute"
is **wrong**, and this cell proves it: the model doesn't use `group` and
discriminates anyway. The enforceable version cites the number:

> *Impact ratio 0.53, below the 0.80 threshold — the automated lending decision has
> disparate impact and, absent human review, is not LOPDP-compliant.*

That is the difference between a compliance document that's a liability and one
that's evidence. **A right you can't measure is a right you can't defend.**

## 5 · The scorecard view — governance controls

| Control | What it addresses | Guarantee (axis 2)? |
|---|---|---|
| "We removed the protected attribute" | disparate impact | **none** — proxy features re-encode it (§4 proves it) |
| Memorization span-scan | PHI leakage | detection only — high recall, needs review |
| **Human-in-the-loop review** | solely-automated decisions | **procedural guarantee** — the LOPDP right, satisfied by design |
| **DP training (DP-SGD)** | memorization + membership | **statable ε bound** (week 11) |
| A **measured** DPIA | all of the above | only as good as its evidence — hence this notebook |

> Governance is where the technical and the legal meet. The LOPDP names the rights;
> the measurements in this notebook are what make them *enforceable*. An unmeasured
> DPIA is the governance equivalent of week 9's system prompt — a request, not a
> control.

## 6 · Your studio deliverable

In `week12/`:

1. **Memorization** — reproduce the PHI extraction; run the deterministic-span scan
   and report what it flags (including false alarms). State which LOPDP right the
   leak violates and how DP-SGD would bound it.
2. **Disparate impact** — compute the impact ratio for the loan model; show that
   dropping the protected attribute does *not* fix it; propose a control and
   measure whether it helps.
3. **Write the DPIA** — for an AI system of your choice (hiring, lending, triage):
   the automated-decision analysis, the memorization/PII risk, and the mitigations,
   each **backed by a measurement**, not an assertion. This is the graded artifact.
4. **Control Scorecard** — classify each mitigation as a *guarantee*, a *procedural
   guarantee*, or *detection only*, with your numbers.

> This week is the course's Latin-American regulatory anchor — keep the LOPDP
> framing; it's what makes the course specific to its students rather than a generic
> US-centric syllabus. It also ties the whole privacy arc together: memorization
> (wk11's attack), the DP guarantee (wk11), and the legal rights those measurements
> enforce (here).